In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
os.chdir('S:/LLC_0028/data/harmonised_all/')

## Symptoms 

In [4]:
symptoms = pd.read_csv('../harmonised_symptom_data/llc_0028_harmsymp_data_v3.csv', index_col=0)

In [6]:
symptoms.shape

(46411, 29)

In [9]:
symptoms=symptoms.drop(['swelling'],axis=1)

## Covariates 

In [10]:
covariates = pd.read_csv('../harmonised_covariates/llc_0028_harmonised_covariates_returned.csv', index_col=0)

In [12]:
covariates = covariates.drop_duplicates(subset = ['LLC_0028_stud_id'])

### Merge 

In [14]:
symp_cov = symptoms.merge(covariates, left_on = 'LLC_0028_stud_id', 
                          right_on='LLC_0028_stud_id', how='left')

In [15]:
symp_cov.shape

(46411, 34)

## Covid status 

In [17]:
dta_loc = '../processed_study_data/'

In [18]:
files = [f for f in os.listdir(dta_loc)]

In [20]:
merged=pd.DataFrame()

for i,f in enumerate(files):
    
    if 'harmonise' in f:
        
        continue
        
    print(f)
    
    df = pd.read_csv(dta_loc + f)
    
    if 'llc_0028_stud_id' in df.columns:

        df = df.rename(columns = {'llc_0028_stud_id':'LLC_0028_stud_id'})

    if 'covid_cat_nofunc' in df.columns:

        df = df.rename(columns = {'covid_cat_nofunc':'covid_status'})
        
    if 'symptoms_date' in df.columns:

        df = df.rename(columns = {'symptoms_date':'symptom_date'})

    if 'functional_limitation' in df.columns:
        
        df = df[['LLC_0028_stud_id', 'covid_status', 'covid_date','symptom_date','functional_limitation']]
        
    else:
        
        df = df[['LLC_0028_stud_id', 'covid_status', 'covid_date','symptom_date']]
    
    #df = df.replace({'LLC_0028_stud_id': id_dict})

    merged = pd.concat([merged,df])

llc_0028_alspacmdat_data_v1.csv
llc_0028_alspacydat_data_v1.csv
llc_0028_bcs70dat_data_v1.csv
llc_0028_bibdat_data_v1.csv
llc_0028_mcsdat_data_v1.csv
llc_0028_ncdsdat_data_v1.csv
llc_0028_nextstepdat_data_v1.csv
llc_0028_nhsd46dat_data_v1.csv
llc_0028_sabredat_data_v1.csv
llc_0028_track19dat_data_v1.csv
llc_0028_twinsdat_data_v1.csv


In [23]:
merged = merged.drop_duplicates(subset = ['LLC_0028_stud_id'])

### Encode covid status 

No longer needed as all extraction in python

In [26]:
merged.covid_status.unique()

array([0, 1, 2], dtype=int64)

### Convert symptom_date to dates 

In [27]:
new_date = []

month2num = {'jan':'01',
             'feb':'02',
             'mar':'03',
            'apr':'04',
             'may':'05',
             'jun':'06',
             'jul':'07',
             'aug':'08',
             'sep':'09',
             'oct':'10',
             'nov':'11',
             'dec':'12'}

for v in merged.symptom_date.values:
    
    if type(v)!=str and np.isnan(v):
        
        new_date.append(v)
    
    elif '/' in v:
        
        new_date.append(v)
        
    else:
        
        day = v[:2]
        year = v[-4:]
        month = month2num[v[2:-4]]
        
        new_date.append(day + '/' + month + '/' + year)
        
merged['symptom_date'] = new_date

## Add EHR covid date 

In [28]:
date = pd.read_csv('../processed_EHR/EHR_infection_first_v2.csv')

### Convert dates 

In [30]:
new_date = []

month2num = {'jan':'01',
             'feb':'02',
             'mar':'03',
            'apr':'04',
             'may':'05',
             'jun':'06',
             'jul':'07',
             'aug':'08',
             'sep':'09',
             'oct':'10',
             'nov':'11',
             'dec':'12'}

for v in date.EHR_COVIDdate.values:
    
    if type(v)!=str and np.isnan(v):
        
        new_date.append(v)
    
    elif '/' in v:
        
        new_date.append(v)
        
    else:
        
        day = v[:2]
        year = v[-4:]
        month = month2num[v[2:-4]]
        
        new_date.append(day + '/' + month + '/' + year)
        
date['EHR_COVIDdate'] = new_date

## Merge all datasets 

In [32]:
final = symp_cov.merge(merged, on = 'LLC_0028_stud_id', how = 'left')

In [34]:
date = date.rename(columns = {'llc_0028_stud_id':'LLC_0028_stud_id'})

final = final.merge(date, on = 'LLC_0028_stud_id', how='left')

## Derive functional limitation categories 

In [36]:
fl = []

studies = final.study.values
for i,v in enumerate(final.functional_limitation.values):
    
    study=studies[i]
    
    #if functional limitation not asked about
    if study not in ['twins','mcs','nextstep','bcs70','nhsd46','ncds','sabre']:
        
        fl.append(-1) #use -1 to represent unknown
        
    elif study=='twins':
        
        if v==0:
            fl.append(0) #no fl
        elif v in [1,2,3]:
            fl.append(1) #1day - 2 weeks
        elif v==4:
            fl.append(2) #2-4 weeks
        elif v==5:
            fl.append(3) #4-12 weeks
        elif v==6:
            fl.append(4) #12+ weeks
        else:
            fl.append(-1) #missing, so mark as unknown
            
    elif study in ['mcs','nextstep','bcs70','nhsd46','ncds','sabre']: #all other cases
            
        if v==1:
            fl.append(0) #no fl
        elif v in [2,3,4]:
            fl.append(1) #1day - 2 weeks
        elif v==5:
            fl.append(2) #2-4 weeks
        elif v==6:
            fl.append(3) #4-12 weeks
        elif v==7:
            fl.append(4) #12+ weeks
        else:
            fl.append(-1) #missing, so mark as unknown
            
final['functional_limitation_cat'] = fl

## Add EHR_status 

In [37]:
ehr_status = []

for i,v in enumerate(final.EHR_COVIDdate.values):
    
    
    if type(v) !=str and np.isnan(v):
        
        ehr_status.append(v)
        
    
    else:
        
        v = pd.to_datetime(v, dayfirst=True, format='%d/%m/%Y')
        symp_date = pd.to_datetime(final.symptom_date.values[i], dayfirst=True, format='%d/%m/%Y')
        date_diff = (symp_date-v).days/7
        
        if date_diff<0:
            
            #had covid after symptom reporting
            ehr_status.append(0)
            
        elif date_diff < 12:
            
            ehr_status.append(1.)
            
        elif date_diff >= 12:
            
            ehr_status.append(2.)
            
        else:
            ehr_status.append(date_diff)
            print(date_diff)

In [38]:
final['ehr_status'] = ehr_status

### Rename covid_status to self_report_status 

In [40]:
final = final.rename(columns = {'covid_status':'self_report_status'})

### Derive combined status 

In [41]:
covid_status = []

for i,v in enumerate(final.self_report_status.values):
    
    if v in [1,2]:
        
        covid_status.append(v)
        
    elif v==0 and ehr_status[i] in [1,2]:
        
        covid_status.append(ehr_status[i])
        
    elif np.isnan(v) and ~np.isnan(ehr_status[i]):
        
        covid_status.append(ehr_status[i])
        
    else:
        
        covid_status.append(v)
        
        
final['covid_status']=covid_status

In [42]:
final.shape

(46411, 43)

### Save file 

In [44]:
final.to_csv('llc_0028_full_harmonised_data_all_v2.csv')

## Check missingness in core symp 

In [46]:
core = ['fever','cough','throat','chest_tight','breath',
        'nose','aches','fatigue','diarrhoea','smell_taste','nausea_vomit',
        'sneezing','headache','concentrating','memory']

### Final study sample -- only studies that reported core symptoms 

In [59]:
sample = final[~(final[core].isnull().any(axis=1))]

In [60]:
sample.shape

(26144, 43)